# Нагрузочный эксперимент для M/M/1

**Цель:** показать нелинейный рост очереди при приближении загрузки к единице.

In [ ]:
using DrWatson
@quickactivate "project"
ENV["GKSwstype"] = "100"
using CSV, DataFrames, Plots, Statistics
include(srcdir("queueing_models.jl"))
using .QueueingModels

name = "02_load_scan"
mkpath(datadir(name)); mkpath(plotsdir(name))
mu = 33.0
lambdas = [10.0, 20.0, 25.0, 28.0, 30.0, 31.5, 32.5]
rows = NamedTuple[]

for (i, lambda) in enumerate(lambdas)
    runs = [simulate_mm1(lambda, mu; horizon=1500.0, seed=202603 + 100i + r) for r in 1:30]
    theory = theoretical_mm1(lambda, mu)
    push!(rows, (lambda=lambda, rho=theory.rho,
        theory_Lq=theory.mean_queue,
        simulation_Lq=mean(getfield.(runs, :mean_queue)),
        simulation_Wq=mean(getfield.(runs, :mean_wait)),
        utilization=mean(getfield.(runs, :utilization))))
end

scan = DataFrame(rows)
CSV.write(datadir(name, "load_scan.csv"), scan)
println("=== Нагрузочный эксперимент M/M/1 ===")
show(scan; allrows=true, allcols=true); println()

default(fontfamily="DejaVu Sans", linewidth=2.4, framestyle=:box, gridalpha=0.22)
p1 = plot(scan.rho, scan.theory_Lq; label="теория", xlabel="Загрузка rho",
    ylabel="Средняя длина очереди", title="Рост очереди при rho → 1",
    size=(1000,620), marker=:circle)
plot!(p1, scan.rho, scan.simulation_Lq; label="моделирование", marker=:diamond)
savefig(p1, plotsdir(name, "load_curve.png"))

p2 = plot(scan.rho, scan.simulation_Wq; label="Wq", xlabel="Загрузка rho",
    ylabel="Среднее ожидание", title="Задержка в зависимости от нагрузки",
    size=(1000,620), marker=:circle, color=:red)
savefig(p2, plotsdir(name, "wait_curve.png"))

При rho, близком к единице, статистическая вариативность возрастает вместе
со средней длиной очереди и временем ожидания.